In [1]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.parsing.metadata_utils import *
from maomao.parsing.integrated_dataset_utils import *

#### Neurotoxic dataset integration and label-consistency analysis
- This notebook integrates peptide-level neurotoxicity annotations from multiple curated sources (NTXpred, NTXpred2, MultiTox, and Peptipedia2.0) into a single, sequence-centric dataset. The inputs are preprocessed CSV files (one per source) containing peptide sequences and neurotoxicity labels.

- All unique sequences across sources are collected to build a pivot table where each row represents a unique peptide and each column corresponds to a source-specific neurotoxicity label. Labels are mapped into the pivot structure using a standardized encoding scheme that separates positive, negative, unlabeled, and unknown annotations.

- Quality control is applied at the sequence level by filtering out peptides containing non-canonical residues and removing sequences outside the global minimum/maximum length constraints. The notebook records the number of sequences before and after each filter, together with summary statistics of the retained length distribution.

- To evaluate cross-source agreement, the notebook computes per-sequence label counts (positive/negative/unlabeled/unknown), derives high-level consistency flags (exclusive positive, exclusive negative, unlabeled-only, or ambiguous), and calculates the percentage of positive vs. negative votes considering only labeled sources. Sequences with conflicting evidence are isolated as ambiguous and stratified by positive-vote percentage to provide a graded confidence view.

- Finally, the curated dataset is exported as non-overlapping subsets (positive, negative, ambiguous) alongside a metadata JSON file summarizing sources, filtering statistics, and label-consistency metrics, enabling reproducible downstream modeling and analysis.

In [2]:
name_task = "toxic_effect_classification"
output_folder = "../../processed_data/integrating_and_cleaning_data/neurotoxic"

# PATH_EXPORT are imported from peptide_toxicity_classifier.constants.
# Update them in constants.py according to the required input and export paths.

- Reading all sources

In [3]:
df_NTXpred_neurotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/NTXpred/processed_neurotoxic_dataset.csv")
df_NTXpred_neurotoxic = df_NTXpred_neurotoxic.rename(columns={"label": "neurotoxic"})

In [4]:
df_NTXpred2_neurotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/NTXpred2/processed_neurotoxic_dataset.csv")
df_NTXpred2_neurotoxic = df_NTXpred2_neurotoxic.rename(columns={"label": "neurotoxic"})

In [5]:
df_MultiTox_neurotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Multitox/processed_neurotoxic_dataset.csv")
df_MultiTox_neurotoxic = df_MultiTox_neurotoxic.rename(columns={"label": "neurotoxic"})

In [6]:
df_peptipedia_neurotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Peptipedia2.0/processed_neurotoxic_dataset.csv")
df_peptipedia_neurotoxic = df_peptipedia_neurotoxic.rename(columns={"label": "neurotoxic"})

In [7]:
df_BiToxNet_neurotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/BiToxNet/processed_neurotoxic_dataset.csv")
df_BiToxNet_neurotoxic = df_BiToxNet_neurotoxic.rename(columns={"label": "neurotoxic"})

- Collecting all sequences for activity

In [8]:
df_list_neurotoxic = [
    df_NTXpred_neurotoxic, df_NTXpred2_neurotoxic, df_MultiTox_neurotoxic,
    df_peptipedia_neurotoxic, df_BiToxNet_neurotoxic
]
unique_sequence_neurotoxic = count_unique_sequence(df_list_neurotoxic)

8701


- Create pivote dataset

In [9]:
df_pivote = create_pivote(unique_sequence_neurotoxic)

- Removing sequences with non canonical residues 

In [10]:
n_before_canon = df_pivote.shape[0]
df_pivote["is_canon"] = df_pivote["sequence"].apply(check_sequence)
n_after_canon = df_pivote[df_pivote["is_canon"]].shape[0]

In [11]:
print(df_pivote["is_canon"].value_counts())
df_pivote = df_pivote[df_pivote["is_canon"]]

is_canon
True     8685
False      16
Name: count, dtype: int64


- Filter sequences by length

In [12]:
df_pivote["length"] = df_pivote["sequence"].str.len()
df_pivote["length"].describe()

count    8685.000000
mean      155.472539
std       226.485798
min         5.000000
25%        57.000000
50%        84.000000
75%       144.000000
max      3933.000000
Name: length, dtype: float64

In [13]:
n_before_length = n_after_canon
df_pivote["filter_length"] = df_pivote["length"].apply(check_length)
n_after_length = df_pivote[df_pivote["filter_length"]].shape[0]

In [14]:
df_pivote["filter_length"].value_counts()

filter_length
False    5502
True     3183
Name: count, dtype: int64

In [15]:
length_series = df_pivote[df_pivote["filter_length"]]["length"]

length_dist = {
    "min": length_series.min(),
    "max": length_series.max(),
    "mean": length_series.mean(),
    "median": length_series.median()
}

In [16]:
df_pivote = df_pivote[df_pivote["filter_length"]]
df_pivote.shape

(3183, 4)

In [17]:
df_pivote = df_pivote.drop(columns=["is_canon", "filter_length", "length"])

In [18]:
df_list_neurotoxic = [("NTXpred", df_NTXpred_neurotoxic), 
                     ("NTXpred2",df_NTXpred2_neurotoxic),
                     ("MultiTox",df_MultiTox_neurotoxic),
                    ("Peptipedia2.0", df_peptipedia_neurotoxic),
                    ("BiToxNet", df_BiToxNet_neurotoxic)
                ]

In [19]:
for source, dataset in df_list_neurotoxic:
    dataset = dataset[["sequence", "neurotoxic"]]
    dataset = dataset.drop_duplicates(subset="sequence")
    mapping = dataset.set_index("sequence")["neurotoxic"]

    # Mapear sin explotar memoria
    df_pivote[source] = (
        df_pivote["sequence"]
        .map(mapping)
        .fillna(999)
        .astype("int16")
    )

In [20]:
df_pivote.head(5)

,sequence,NTXpred,NTXpred2,MultiTox,Peptipedia2.0,BiToxNet
2,MKSTLMTASVLILVLLSIVDYASVYAEFIDSEISLERQWINACFNV...,999,1,1,999,1
4,RIKKPIFAFPRF,999,1,999,999,1
6,MRYTDSRKLTPETDANHKTASPQPIRRISSQTLLGPDGKLIIDHDG...,999,999,999,999,0
8,MTPPENKNLVQENKELIQEVLKA,999,0,999,999,0
11,GRDAYIAQPENCVYECAKNSYCNDLCTKNGAKSGYCQWLGRWGNAC...,1,999,999,999,999


- Working with pivote for detecting ambiguous sequences 

In [21]:
df_pivote = process_count_labels(df_pivote)  # Verify the consistency of the labels by source

In [22]:
df_pivote["negative"].value_counts() # Includes sources labeled as nevative (0) and unlabeled (2)

negative
True     1700
False    1483
Name: count, dtype: int64

In [23]:
df_pivote["exclusive_0"].value_counts()

exclusive_0
True     1700
False    1483
Name: count, dtype: int64

In [24]:
df_pivote["positive"].value_counts() # Includes sources labeled as positive (1) and unlabeled (2)

positive
False    1702
True     1481
Name: count, dtype: int64

In [25]:
df_pivote["exclusive_1"].value_counts()

exclusive_1
False    1702
True     1481
Name: count, dtype: int64

In [26]:
df_pivote["only_unlabel"].value_counts() # Includes only sources unlabeled (2)

only_unlabel
False    3183
Name: count, dtype: int64

In [27]:
df_pivote.sort_values(by="percentage_1", ascending=False)

,sequence,NTXpred,NTXpred2,MultiTox,Peptipedia2.0,BiToxNet,counts_1,counts_0,counts_unlabel,counts_unknown,positive,negative,exclusive_1,exclusive_0,only_unlabel,percentage_0,percentage_1
31,YCQKFLWTCDTERKCCEDMVCELWCKLEK,999,1,999,1,1,3,0,0,2,True,False,True,False,False,0.0,100.0
8698,TFINVKCTSPKQCLKPCKDLYGPHAGEKCMNGKCKCYKP,999,1,999,999,1,2,0,0,3,True,False,True,False,False,0.0,100.0
2,MKSTLMTASVLILVLLSIVDYASVYAEFIDSEISLERQWINACFNV...,999,1,1,999,1,3,0,0,2,True,False,True,False,False,0.0,100.0
4,RIKKPIFAFPRF,999,1,999,999,1,2,0,0,3,True,False,True,False,False,0.0,100.0
41,KDKENCIGKHHECTDDRDSCCKGKLFRYQCQCFKVIDGKKETKRCA...,999,999,999,999,1,1,0,0,4,True,False,True,False,False,0.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2732,APVPGLSPFRVV,999,0,999,999,0,0,2,0,3,False,True,False,True,False,100.0,0.0
2734,YLYQWLGAPAPYPDPLEPKREVCELNPDCDELADHIGFQEAYRRFYGPV,999,0,999,999,0,0,2,0,3,False,True,False,True,False,100.0,0.0
2736,MKFFSVVTVFVFGLLALANAVPLSPDPGNVVINGDCKYCNVHGGK,999,0,999,999,0,0,2,0,3,False,True,False,True,False,100.0,0.0
2739,MPITDPVKLQIVYNRVLNKKVCRKCGALNPPTATKCRRCKSKNLRP...,999,999,999,999,0,0,1,0,4,False,True,False,True,False,100.0,0.0


- Splitting data into only negative, only positive, and with amiguous data

In [28]:
negative = df_pivote[df_pivote["negative"]]

In [29]:
only_negative = df_pivote[df_pivote["exclusive_0"]]

In [30]:
positive = df_pivote[df_pivote["positive"]]

In [31]:
only_positive = df_pivote[df_pivote["exclusive_1"]]

In [32]:
only_unlabel = df_pivote[df_pivote["only_unlabel"]]

In [33]:
df_ambiguous = df_pivote[(df_pivote["positive"] == False) & (df_pivote["negative"] == False) & (df_pivote["only_unlabel"] == False)]

- Processing ambiguous data

In [34]:
df_ambiguous = categorize_percentage(df_ambiguous)

In [35]:
df_ambiguous["Category_pbb"].value_counts()

Category_pbb
30-40    2
Name: count, dtype: int64

- Working with metada

In [36]:
seq_stats = {
    "canonical": {
        "before": n_before_canon,
        "after": n_after_canon
    },
    "length": {
        "before": n_before_length,
        "after": n_after_length,
        "min": MIN_LENGTH_SEQUENCE,
        "max": MAX_LENGTH_SEQUENCE
    },
    "length_dist": length_dist
}

metadata = build_dataset_metadata(
    task="neurotoxic",
    source_list=df_list_neurotoxic,
    pivote_df=df_pivote,
    outputs={
        "only_positive": only_positive,
        "only_negative": only_negative,
        "ambiguous": df_ambiguous
    },
    seq_stats=seq_stats,
    filters={
        "canonical_residues": True,
        "length_filter": True
    }
)
metadata

{'task': 'neurotoxic',
 'generated_at': '2026-09-04T20:36:32.891793',
 'sources': {'n_unique_sequences': {'NTXpred': 916,
   'NTXpred2': 3300,
   'MultiTox': 809,
   'Peptipedia2.0': 577,
   'BiToxNet': 7780}},
 'filters': {'canonical_residues': {'applied': True},
  'length_filter': {'applied': True, 'min_length': 5, 'max_length': 70}},
 'sequence_statistics': {'canonical_filter': {'before': 8701, 'after': 8685},
  'length_filter': {'before': 8685, 'after': 3183},
  'length_distribution': {'min': 5, 'max': 70, 'mean': 44.08, 'median': 43.0}},
 'statistics': {'total_sequences_final': 3183,
  'positive': {'positive_and_unlabel': 1481, 'only_positive': 1481},
  'negative': {'negative_and_unlabel': 1700, 'only_negative': 1700},
  'only_unlabel': 0,
  'ambiguous': {'n_sequences': 2,
   'category_pbb': {'description': 'Distribution of ambiguous sequences based on the percentage of positive annotations across sources',
    'categories_definition': 'Bins represent the percentage of sources lab

- Exporting data

In [37]:
os.makedirs(output_folder, exist_ok=True)

In [38]:
with open(f"{output_folder}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [39]:
negative.shape

(1700, 17)

In [40]:
only_negative.shape

(1700, 17)

In [41]:
positive.shape

(1481, 17)

In [42]:
only_positive.shape

(1481, 17)

In [43]:
only_unlabel.shape

(0, 17)

In [44]:
df_ambiguous.shape

(2, 18)

In [45]:
negative.to_csv(f"{output_folder}/negative.csv", index=False)

In [46]:
positive.to_csv(f"{output_folder}/positive.csv", index=False)

In [47]:
df_ambiguous.to_csv(f"{output_folder}/ambiguous_data.csv", index=False)